In [1]:


import numpy as np
import pandas as pd

print("NumPy version: ",  np.__version__)
print("Pandas version:",  pd.__version__)

NumPy version:  2.5.1
Pandas version: 3.0.3


In [2]:

orders    = pd.read_csv("orders.csv")
products  = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")

print("=== ORDERS ===")
orders.info()
orders.describe()



=== ORDERS ===
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      1000 non-null   int64
 1   customer_id   1000 non-null   int64
 2   product_id    1000 non-null   int64
 3   quantity      1000 non-null   int64
 4   discount_pct  1000 non-null   int64
 5   order_date    1000 non-null   str  
dtypes: int64(5), str(1)
memory usage: 47.0 KB


,order_id,customer_id,product_id,quantity,discount_pct
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1500.500000,225.782000,310.455000,3.013000,7.540000
std,288.819436,14.372858,5.796514,1.420158,7.609619
min,1001.000000,201.000000,301.000000,1.000000,0.000000
25%,1250.750000,213.000000,305.000000,2.000000,0.000000
50%,1500.500000,226.000000,310.000000,3.000000,5.000000
75%,1750.250000,238.000000,315.000000,4.000000,15.000000
max,2000.000000,250.000000,320.000000,5.000000,20.000000


In [3]:


quantities_arr = orders["quantity"].values       # .values converts to NumPy array
prices_arr     = products["unit_price"].values

print("Total units ordered:",  np.sum(quantities_arr))
print("Avg qty per order:  ",  np.mean(quantities_arr).round(2))
print("Largest single order:", np.max(quantities_arr))
print("Cheapest product: Rs.",  np.min(prices_arr).round(2))
print("Most expensive:   Rs.",  np.max(prices_arr).round(2))
print("Average price:    Rs.",  np.mean(prices_arr).round(2))

# Revenue potential before discounts
avg_order_revenue = np.mean(prices_arr) * np.mean(quantities_arr)
print(f"Avg order revenue estimate: Rs.{avg_order_revenue:,.0f}")

# Percentile breakdown
for p in [25, 50, 75, 90]:
    print(f"  {p}th percentile price: Rs.{np.percentile(prices_arr, p):,.2f}")

#

# "From the percentile output — 75th percentile price is around Rs.3,500.
#  What does this tell the business?"
# → Answer: 75% of products are priced below Rs.3,500.
#   Only 25% are premium tier.


Total units ordered: 3013
Avg qty per order:   3.01
Largest single order: 5
Cheapest product: Rs. 302.45
Most expensive:   Rs. 4562.09
Average price:    Rs. 2081.79
Avg order revenue estimate: Rs.6,272
  25th percentile price: Rs.1,327.81
  50th percentile price: Rs.1,778.45
  75th percentile price: Rs.2,636.58
  90th percentile price: Rs.3,644.15


In [4]:





orders_dirty = orders.copy()  

# Inject 50 null discount values (~5% of rows)
null_idx_disc = np.random.choice(orders_dirty.index, size=50, replace=False)
orders_dirty.loc[null_idx_disc, "discount_pct"] = np.nan

# Inject 30 null quantity values (~3% of rows)
null_idx_qty  = np.random.choice(orders_dirty.index, size=30, replace=False)
orders_dirty.loc[null_idx_qty, "quantity"] = np.nan

print(f"Dirty dataset shape  : {orders_dirty.shape}")
print(f"Nulls per column:\n{orders_dirty.isnull().sum()}")



Dirty dataset shape  : (1000, 6)
Nulls per column:
order_id         0
customer_id      0
product_id       0
quantity        30
discount_pct    50
order_date       0
dtype: int64


In [5]:
# ============================================================
# Step 7: Data Cleaning (20 min)
# 
# ============================================================

# 🖊️ TYPE — Step 7: Remove duplicates and fill nulls

# 1. Remove duplicates
orders_clean = orders_dirty.drop_duplicates()
print(f"After removing duplicates: {orders_clean.shape}")   # Back to (1000, 6)

# 2. Inspect remaining nulls
print(f"Nulls before fix:\n{orders_clean.isnull().sum()}")

# Fill discount nulls with 0
# Business logic: if no discount was recorded, none was applied
orders_clean["discount_pct"] = orders_clean["discount_pct"].fillna(0)

# 
median_qty = orders_clean["quantity"].median()
orders_clean["quantity"] = orders_clean["quantity"].fillna(median_qty)

print(f"Nulls after fix:\n{orders_clean.isnull().sum()}")
print(f"Final clean shape: {orders_clean.shape}")




After removing duplicates: (1000, 6)
Nulls before fix:
order_id         0
customer_id      0
product_id       0
quantity        30
discount_pct    50
order_date       0
dtype: int64
Nulls after fix:
order_id        0
customer_id     0
product_id      0
quantity        0
discount_pct    0
order_date      0
dtype: int64
Final clean shape: (1000, 6)


In [6]:
# ============================================================
# Step 8: Date Handling and Feature Engineering (20 min)
# 

orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])
customers["join_date"]     = pd.to_datetime(customers["join_date"])

print("order_date dtype:", orders_clean["order_date"].dtype)
# datetime64[us] — now a proper datetime

# Extract 5 analysis columns from one date
orders_clean["year"]       = orders_clean["order_date"].dt.year
orders_clean["month"]      = orders_clean["order_date"].dt.month
orders_clean["month_name"] = orders_clean["order_date"].dt.strftime("%b")
orders_clean["day_name"]   = orders_clean["order_date"].dt.day_name()
orders_clean["quarter"]    = orders_clean["order_date"].dt.quarter

orders_clean[["order_date","year","month","month_name",
               "day_name","quarter"]].head(8)




order_date dtype: datetime64[us]


,order_date,year,month,month_name,day_name,quarter
0,2024-05-26,2024,5,May,Sunday,2
1,2024-05-27,2024,5,May,Monday,2
2,2024-12-17,2024,12,Dec,Tuesday,4
3,2024-07-17,2024,7,Jul,Wednesday,3
4,2024-11-03,2024,11,Nov,Sunday,4
5,2024-05-07,2024,5,May,Tuesday,2
6,2024-02-08,2024,2,Feb,Thursday,1
7,2024-12-03,2024,12,Dec,Tuesday,4


In [7]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 8b: Customer tenure feature
#.

# How long has each customer been with ShopEase?
today = pd.Timestamp("2024-12-31")

customers["tenure_days"]  = (today - customers["join_date"]).dt.days
customers["tenure_years"] = (customers["tenure_days"] / 365).round(1)

customers[["customer_id","join_date","tenure_days","tenure_years"]].head()



,customer_id,join_date,tenure_days,tenure_years
0,201,2024-04-19,256,0.7
1,202,2022-08-18,866,2.4
2,203,2024-10-28,64,0.2
3,204,2021-09-04,1214,3.3
4,205,2024-01-25,341,0.9


In [8]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 8c: Age group segmentation with apply()
# ------------------------------------------------------------

# ── Method 1: Define a function, apply it to each value ───────────────────
def age_group(age):
    if age < 25:   return "18-24"
    elif age < 35: return "25-34"
    elif age < 45: return "35-44"
    else:          return "45+"

customers["age_group"] = customers["age"].apply(age_group)
print(customers["age_group"].value_counts())

# ── Method 2: Lambda — same result, more compact ──────────────────────────
customers["age_group"] = customers["age"].apply(
    lambda x: "18-24" if x < 25 else "25-34" if x < 35 else "35-44" if x < 45 else "45+"
)

# ── Method 3: np.select — FASTEST for large datasets ─────────────────────
conditions = [customers["age"] < 25, customers["age"] < 35, customers["age"] < 45]
choices    = ["18-24", "25-34", "35-44"]
customers["age_group"] = np.select(conditions, choices, default="45+")




age_group
45+      17
35-44    16
25-34    10
18-24     7
Name: count, dtype: int64


In [9]:
# ============================================================
# Step 9: Build the Master Analysis Table (10 min)

master = pd.merge(orders_clean, products,
                  how="inner", on="product_id")
print("After orders + products:", master.shape)

# Merge with customer details
master = pd.merge(master,
                  customers[["customer_id","age","gender",
                              "city","age_group","tenure_years"]],
                  how="inner", on="customer_id")
print("After + customers      :", master.shape)
print("Columns:", list(master.columns))

After orders + products: (1000, 15)
After + customers      : (1000, 20)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'discount_pct', 'order_date', 'year', 'month', 'month_name', 'day_name', 'quarter', 'product_name', 'category', 'unit_price', 'unit_cost', 'age', 'gender', 'city', 'age_group', 'tenure_years']


In [10]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 9b: Calculate revenue and profit columns
# ------------------------------------------------------------

# 

# 🔢 Formulas:
#    gross_revenue = quantity × unit_price
#    discount_amt  = gross_revenue × (discount_pct / 100)
#    net_revenue   = gross_revenue − discount_amt
#    cost          = quantity × unit_cost
#    profit        = net_revenue − cost
#    profit_margin = (profit / net_revenue) × 100

# All 1000 rows calculated simultaneously — no loop
master["gross_revenue"] = master["quantity"] * master["unit_price"]
master["discount_amt"]  = master["gross_revenue"] * (master["discount_pct"] / 100)
master["net_revenue"]   = master["gross_revenue"] - master["discount_amt"]
master["cost"]          = master["quantity"] * master["unit_cost"]
master["profit"]        = master["net_revenue"] - master["cost"]
master["profit_margin"] = np.round((master["profit"] / master["net_revenue"]) * 100, 2)

master[["order_id","quantity","unit_price","discount_pct",
        "gross_revenue","net_revenue","profit","profit_margin"]].head(5)




,order_id,quantity,unit_price,discount_pct,gross_revenue,net_revenue,profit,profit_margin
0,1001,3.0,1732.47,0.0,5197.41,5197.4100,2477.5500,47.67
1,1002,3.0,873.81,0.0,2621.43,2621.4300,1378.7100,52.59
2,1003,5.0,2257.60,10.0,11288.00,10159.2000,4276.4500,42.09
3,1004,5.0,3554.45,5.0,17772.25,16883.6375,6145.4375,36.40
4,1005,1.0,873.81,20.0,873.81,699.0480,284.8080,40.74


In [11]:


print("=" * 55)
print("  SHOPEASE INDIA — 2024 ANNUAL SUMMARY")
print("=" * 55)
print(f"  Total Orders      : {master['order_id'].nunique():,}")
print(f"  Total Customers   : {master['customer_id'].nunique():,}")
print(f"  Gross Revenue     : Rs.{master['gross_revenue'].sum():,.0f}")
print(f"  Total Discounts   : Rs.{master['discount_amt'].sum():,.0f}")
print(f"  Net Revenue       : Rs.{master['net_revenue'].sum():,.0f}")
print(f"  Total Profit      : Rs.{master['profit'].sum():,.0f}")
print(f"  Avg Order Value   : Rs.{master['net_revenue'].mean():,.0f}")
print(f"  Avg Profit Margin : {master['profit_margin'].mean():.1f}%")
print("=" * 55)



  SHOPEASE INDIA — 2024 ANNUAL SUMMARY
  Total Orders      : 1,000
  Total Customers   : 50
  Gross Revenue     : Rs.6,447,232
  Total Discounts   : Rs.477,999
  Net Revenue       : Rs.5,969,233
  Total Profit      : Rs.2,431,171
  Avg Order Value   : Rs.5,969
  Avg Profit Margin : 41.4%


In [12]:


monthly = master.groupby(["month","month_name"]).agg(
    total_orders  = ("order_id",    "count"),
    net_revenue   = ("net_revenue", "sum"),
    total_profit  = ("profit",      "sum"),
    avg_order_val = ("net_revenue", "mean")
).reset_index().sort_values("month")

monthly["net_revenue"]   = monthly["net_revenue"].round(0)
monthly["avg_order_val"] = monthly["avg_order_val"].round(0)

print(monthly[["month_name","total_orders",
               "net_revenue","avg_order_val"]].to_string(index=False))

# Automatically identify best and worst months
best  = monthly.loc[monthly["net_revenue"].idxmax(), "month_name"]
worst = monthly.loc[monthly["net_revenue"].idxmin(), "month_name"]
print(f"Best Month: {best}  |  Worst Month: {worst}")



month_name  total_orders  net_revenue  avg_order_val
       Jan            83     521785.0         6287.0
       Feb            84     470929.0         5606.0
       Mar            64     425537.0         6649.0
       Apr            94     577002.0         6138.0
       May           103     582153.0         5652.0
       Jun            89     550789.0         6189.0
       Jul            65     402419.0         6191.0
       Aug            87     473852.0         5447.0
       Sep            76     439556.0         5784.0
       Oct            92     624821.0         6792.0
       Nov            81     450199.0         5558.0
       Dec            82     450191.0         5490.0
Best Month: Oct  |  Worst Month: Jul


In [13]:
# ============================================================
# Steps 12-13: Category and City Analysis — Q2 & Q3 (12 min)
# 

# ------------------------------------------------------------
# 🖊️ TYPE — Step 12: Category performance — answering Q2
# ------------------------------------------------------------

category_perf = master.groupby("category").agg(
    total_orders = ("order_id",       "count"),
    units_sold   = ("quantity",       "sum"),
    net_revenue  = ("net_revenue",    "sum"),
    total_profit = ("profit",         "sum"),
    avg_margin   = ("profit_margin",  "mean")
).reset_index().sort_values("net_revenue", ascending=False)

print(category_perf.to_string(index=False))

print(f"Top by Revenue : {category_perf.iloc[0]['category']}")
print(f"Top by Margin  : {category_perf.loc[category_perf['avg_margin'].idxmax(), 'category']}")

# ===========================================================================
# 💡 EXPLAIN — The revenue vs margin insight — pause here
# ===========================================================================

# "The top category by revenue is NOT always the most profitable one."
# "High revenue with thin margins might be worse for the business than
#  moderate revenue with fat margins."
# "This is a real strategic decision: do you optimise for volume or profit?"
# "You just found this insight in 8 lines of code.
#  In Excel, this would take an hour."


      category  total_orders  units_sold  net_revenue  total_profit  avg_margin
         Books           376      1139.0 2323294.7985   869124.7385   39.127527
   Electronics           165       525.0 1551756.2710   690892.6810   43.941939
        Sports           252       744.0 1158500.3275   483886.9475   43.301786
Home & Kitchen           110       316.0  516793.9170   237006.7570   45.468182
       Fashion            97       293.0  418887.6670   150260.0470   36.195155
Top by Revenue : Books
Top by Margin  : Home & Kitchen


In [14]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 13: City-wise analysis — part of Q3
# ------------------------------------------------------------

# 💡 nunique() vs count():
#    count()   → total rows (orders)
#    nunique() → distinct IDs (unique customers)
#    Always use nunique() for customer counts!

city_analysis = master.groupby("city").agg(
    unique_customers = ("customer_id", "nunique"),
    total_orders     = ("order_id",    "count"),
    net_revenue      = ("net_revenue", "sum"),
    avg_order_value  = ("net_revenue", "mean"),
    total_profit     = ("profit",      "sum")
).reset_index().sort_values("net_revenue", ascending=False)

# Derived metric: engagement per customer
city_analysis["orders_per_cust"] = (
    city_analysis["total_orders"] / city_analysis["unique_customers"]
).round(1)

print(city_analysis.to_string(index=False))




     city  unique_customers  total_orders  net_revenue  avg_order_value  total_profit  orders_per_cust
   Mumbai                11           223 1220483.6805      5473.020989   478765.0305             20.3
Ahmedabad                 8           155  949102.7095      6123.243287   398362.9895             19.4
  Kolkata                 7           150  906938.5810      6046.257207   376593.8110             21.4
  Chennai                 7           139  875064.2130      6295.425993   355018.0130             19.9
Bengaluru                 7           140  792662.8160      5661.877257   327817.7260             20.0
Hyderabad                 5            93  622527.4855      6693.843930   260593.4455             18.6
    Delhi                 3            62  378879.0445      6110.952331   144299.3645             20.7
     Pune                 2            38  223574.4510      5883.538184    89720.7910             19.0


In [15]:
# ============================================================
# Steps 14-15: Customer Segments and Lifetime Value — Q3 & Q6 (12 min)
# ⏱️ TIMING — 12 min | Step 15 is the most complex query.
#             Explain the 2-step pattern before typing.
# ============================================================

# 🖊️ TYPE — Step 14: Age group × gender — deep dive into Q3

# 🎯 This is how marketers make targeting decisions:
#    Which age-gender combo has the highest avg_spend?
#    → That segment gets premium ad spend.

segment = master.groupby(["age_group","gender"]).agg(
    customers = ("customer_id", "nunique"),
    orders    = ("order_id",    "count"),
    revenue   = ("net_revenue", "sum"),
    avg_spend = ("net_revenue", "mean")
).reset_index().sort_values("revenue", ascending=False)

segment["revenue"]   = segment["revenue"].round(0)
segment["avg_spend"] = segment["avg_spend"].round(0)

print(segment.to_string(index=False))


age_group gender  customers  orders   revenue  avg_spend
      45+ Female         13     244 1450324.0     5944.0
    35-44   Male          8     155 1029166.0     6640.0
    35-44 Female          8     167  905820.0     5424.0
    25-34 Female          7     151  882071.0     5842.0
    18-24 Female          6     123  833365.0     6775.0
      45+   Male          4      75  412946.0     5506.0
    25-34   Male          3      65  358138.0     5510.0
    18-24   Male          1      20   97402.0     4870.0


In [16]:
# ------------------------------------------------------------
# 💡 EXPLAIN — Before Step 15 — set up the two-step mental model
# ------------------------------------------------------------

# "Step 15 has two parts and students often get lost. Explain before typing:"

# Part 1: groupby on customer_id
#   → Collapses all orders into ONE row per customer
#   → Each row = one customer's lifetime summary:
#     total orders, total revenue, first/last purchase

# Part 2: merge with customers table
#   → Adds back name, city, gender, age
#   → The groupby REMOVES all customer details. The merge PUTS THEM BACK.

# "Why not include them in the groupby?
#  Because city and gender don't aggregate — they just describe."

# ------------------------------------------------------------
# 🖊️ TYPE — Step 15: Top 10 customers by lifetime value — answering Q6
# ------------------------------------------------------------

customer_ltv = master.groupby("customer_id").agg(
    total_orders  = ("order_id",    "count"),
    total_revenue = ("net_revenue", "sum"),
    total_profit  = ("profit",      "sum"),
    avg_order_val = ("net_revenue", "mean"),
    first_order   = ("order_date",  "min"),
    last_order    = ("order_date",  "max")
).reset_index()

# Merge back with customer details to get city, gender, age
customer_ltv = pd.merge(customer_ltv,
                        customers[["customer_id","age","gender","city"]],
                        on="customer_id")

customer_ltv = customer_ltv.sort_values("total_revenue", ascending=False)
top10 = customer_ltv.head(10)
top10["total_revenue"] = top10["total_revenue"].round(0)

print("TOP 10 CUSTOMERS BY LIFETIME VALUE:")
print(top10[["customer_id","city","gender","age",
             "total_orders","total_revenue","avg_order_val"]].to_string(index=False))

# ===========================================================================
# 🤖 AI PROMPT — Use AI for a real business discussion
# ===========================================================================

# Prompt: "Our top customer has the highest total revenue but only 12 orders.
#          Another customer has 25 orders but 40% less revenue.
#          Which one is more valuable to us long-term, and why?"

# Let students read the AI response, then debate:
# "Do you agree with AI? What did it miss?"
# → This is the kind of discussion that happens in actual product
#   and strategy meetings.


TOP 10 CUSTOMERS BY LIFETIME VALUE:
 customer_id      city gender  age  total_orders  total_revenue  avg_order_val
         229     Delhi Female   42            29       200657.0    6919.223448
         237   Chennai Female   20            25       199434.0    7977.361360
         245 Hyderabad   Male   42            23       183441.0    7975.707957
         225 Bengaluru Female   50            26       165725.0    6374.027365
         208   Kolkata   Male   36            24       164962.0    6873.436479
         244 Ahmedabad Female   21            22       163818.0    7446.265932
         234   Kolkata Female   32            29       153491.0    5292.788931
         203   Chennai Female   32            21       151063.0    7193.460310
         214 Ahmedabad Female   57            20       141774.0    7088.681975
         224 Bengaluru   Male   38            20       137792.0    6889.592775


In [17]:
# ============================================================
# Steps 16-18: Day-of-Week, Discount Impact, Quarterly (12 min)
# ==========================================================

# "Step 16: What day of the week gets the most orders?"
# "You have day_name in master. Write the groupby yourself — 3 minutes,
#  then we compare."
# → Walk the room. After 3 min, show the solution.

# ------------------------------------------------------------
# 🖊️ TYPE — Step 16: Day-of-week sales pattern
# ------------------------------------------------------------

day_order = ["Monday","Tuesday","Wednesday","Thursday",
             "Friday","Saturday","Sunday"]

dow = master.groupby("day_name").agg(
    orders      = ("order_id",    "count"),
    net_revenue = ("net_revenue", "sum"),
    avg_order   = ("net_revenue", "mean")
# reindex forces natural day order instead of alphabetical
).reindex(day_order).reset_index()

busiest_day = dow.loc[dow["orders"].idxmax(), "day_name"]

print(dow.to_string(index=False))
print(f"Busiest Day: {busiest_day}")

# ===========================================================================
# 💡 EXPLAIN — Why .reindex() is a real-world trick
# ===========================================================================

# "Without reindex(), Pandas sorts day names alphabetically:
#  Friday, Monday, Saturday..."
# "reindex(day_order) forces a custom sort order."
# "You will use this pattern any time you need a non-alphabetical sort:
#  months, custom priority levels, age groups."


 day_name  orders  net_revenue   avg_order
   Monday     145  881657.9075 6080.399362
  Tuesday     143  817244.0340 5714.993245
Wednesday     117  651641.7530 5569.587632
 Thursday     139  879596.9070 6328.035302
   Friday     159  954526.1615 6003.309192
 Saturday     151  941965.6400 6238.183046
   Sunday     146  842600.5780 5771.236836
Busiest Day: Friday


In [18]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 17: Discount impact analysis — new concept: pd.cut()
# ------------------------------------------------------------

# 💡 pd.cut() bins continuous values into labelled groups.
#    Without it, groupby would make one group per unique discount value
#    (0, 5, 10, 15, 20) — which already happens to be clean here.
#    In real data, you might have 0.0, 0.5, 1.2, 2.7... pd.cut() solves that.

# ❓ Key question: Do bigger discounts actually generate more profit?
#    Or do they just eat into margins?

bins   = [-1, 0, 5, 10, 15, 20]
labels = ["No Discount", "5%", "10%", "15%", "20%"]
master["discount_bucket"] = pd.cut(master["discount_pct"],
                                    bins=bins, labels=labels)

disc_analysis = master.groupby("discount_bucket", observed=True).agg(
    orders        = ("order_id",      "count"),
    net_revenue   = ("net_revenue",   "sum"),
    avg_margin    = ("profit_margin", "mean"),
    total_profit  = ("profit",        "sum")
).reset_index().round(0)

print("Discount Impact on Revenue and Profit:")
print(disc_analysis.to_string(index=False))




Discount Impact on Revenue and Profit:
discount_bucket  orders  net_revenue  avg_margin  total_profit
    No Discount     437    2697261.0        46.0     1206801.0
             5%     125     787345.0        44.0      336798.0
            10%     147     857490.0        41.0      339177.0
            15%     144     890809.0        35.0      310777.0
            20%     147     736328.0        33.0      237618.0


In [19]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 18: Quarterly performance
# ------------------------------------------------------------

# 🎯 Quarters are the standard unit of business reporting.
#    Q1 vs Q4 comparison tells you if the business is growing.

quarterly = master.groupby("quarter").agg(
    orders      = ("order_id",     "count"),
    net_revenue = ("net_revenue",  "sum"),
    profit      = ("profit",       "sum"),
    avg_margin  = ("profit_margin","mean")
).reset_index()

quarterly.columns = ["Quarter","Orders","Net Revenue","Profit","Avg Margin"]
quarterly[["Net Revenue","Profit"]] = quarterly[["Net Revenue","Profit"]].round(0)
quarterly["Avg Margin"] = quarterly["Avg Margin"].round(1)

print(quarterly.to_string(index=False))


 Quarter  Orders  Net Revenue   Profit  Avg Margin
       1     231    1418251.0 590605.0        41.5
       2     286    1709944.0 688277.0        41.1
       3     228    1315827.0 515189.0        40.9
       4     255    1525211.0 637101.0        42.0


In [20]:
# ============================================================
# Steps 19-20: Product Stars and Executive Summary (12 min)
# ⏱
# ============================================================

# 🖊️ TYPE — Step 19: Star products and slow movers

# 🎯 Stars      → high revenue, high profit → increase inventory, promote more
#    Slow movers → low revenue, low profit  → consider discounting or delisting

product_perf = master.groupby(["product_id","product_name","category"]).agg(
    units_sold  = ("quantity",    "sum"),
    net_revenue = ("net_revenue", "sum"),
    profit      = ("profit",      "sum")
).reset_index().sort_values("net_revenue", ascending=False)

print("⭐ TOP 5 PRODUCTS (by Revenue):")
print(product_perf.head(5)[["product_name","category",
                              "units_sold","net_revenue","profit"]].to_string(index=False))

print("\n🐢 BOTTOM 5 PRODUCTS (Slow Movers):")
print(product_perf.tail(5)[["product_name","category",
                              "units_sold","net_revenue","profit"]].to_string(index=False))

# ===========================================================================
# 💡 EXPLAIN — The business value of slow mover detection
# ===========================================================================

# "Bottom 5 products are candidates for three actions:
#  discontinue, discount to clear stock, or reposition."
# "Finding these with 8 lines of code vs manually scanning thousands
#  of Excel rows — that is the analyst's value."
# "A data analyst who can answer 'which products should we discontinue?'
#  in 5 minutes is worth hiring."


⭐ TOP 5 PRODUCTS (by Revenue):
product_name    category  units_sold  net_revenue      profit
 Product_303 Electronics       195.0  805484.4490 441374.5990
 Product_307       Books       139.0  586000.4605 196846.3305
 Product_309 Electronics       162.0  504129.1050 125271.0450
 Product_316       Books       149.0  500999.7275 181001.3675
 Product_314      Sports       169.0  436005.6330 196868.9430

🐢 BOTTOM 5 PRODUCTS (Slow Movers):
product_name category  units_sold  net_revenue     profit
 Product_301    Books       129.0  160328.0695 63218.1595
 Product_319  Fashion       143.0  145208.1070 56928.4870
 Product_313    Books       144.0  142772.3460 79383.5460
 Product_305   Sports       137.0  111192.3225 54441.4425
 Product_310   Sports       135.0   38048.2100 18447.5600


In [21]:
# ------------------------------------------------------------
# 🖊️ TYPE — Step 20: Final executive summary — NumPy returns
# ------------------------------------------------------------

# Pull raw NumPy arrays from Pandas columns
rev_arr    = master["net_revenue"].values
profit_arr = master["profit"].values
qty_arr    = master["quantity"].values

print("=" * 60)
print("  SHOPEASE INDIA — EXECUTIVE SUMMARY (FY 2024)")
print("=" * 60)
print(f"  Total Transactions      : {len(rev_arr):,}")
print(f"  Total Net Revenue       : Rs.{np.sum(rev_arr):>15,.0f}")
print(f"  Total Profit            : Rs.{np.sum(profit_arr):>15,.0f}")
print(f"  Avg Order Value         : Rs.{np.mean(rev_arr):>15,.0f}")
print(f"  Median Order Value      : Rs.{np.median(rev_arr):>15,.0f}")
print(f"  Std Dev (Order Values)  : Rs.{np.std(rev_arr):>15,.0f}")
print(f"  Overall Profit Margin   :  {(np.sum(profit_arr)/np.sum(rev_arr)*100):>13.1f}%")
print(f"  Total Units Sold        : {np.sum(qty_arr):>16,}")
print("=" * 60)

# Revenue distribution
p25 = np.percentile(rev_arr, 25)
p75 = np.percentile(rev_arr, 75)
print(f"  25th Percentile         : Rs.{p25:,.0f}")
print(f"  75th Percentile         : Rs.{p75:,.0f}")
print(f"  IQR                     : Rs.{(p75-p25):,.0f}")
print("=" * 60)



  SHOPEASE INDIA — EXECUTIVE SUMMARY (FY 2024)
  Total Transactions      : 1,000
  Total Net Revenue       : Rs.      5,969,233
  Total Profit            : Rs.      2,431,171
  Avg Order Value         : Rs.          5,969
  Median Order Value      : Rs.          4,699
  Std Dev (Order Values)  : Rs.          4,641
  Overall Profit Margin   :           40.7%
  Total Units Sold        :          3,017.0
  25th Percentile         : Rs.2,656
  75th Percentile         : Rs.7,828
  IQR                     : Rs.5,172
